# Exercícios - Aula 6

Rodrigo Gimenes Rodrigues - CP3071782


## Exercício 1: Caçador de Outliers

Carregar o tips do Seaborn, achar outliers de total_bill pelo IQR, tirar esses pontos e comparar a média antes/depois.


In [1]:
import seaborn as sns
import pandas as pd
import numpy as np

df = sns.load_dataset('tips')

q1 = df['total_bill'].quantile(0.25)
q3 = df['total_bill'].quantile(0.75)
iqr = q3 - q1
lim_inf = q1 - 1.5 * iqr
lim_sup = q3 + 1.5 * iqr

print('Q1:', round(q1, 2), 'Q3:', round(q3, 2), 'IQR:', round(iqr, 2))
print('Limites:', round(lim_inf, 2), 'a', round(lim_sup, 2))

outliers = df[(df['total_bill'] < lim_inf) | (df['total_bill'] > lim_sup)]
print('Qtd outliers:', len(outliers))

df_sem = df[(df['total_bill'] >= lim_inf) & (df['total_bill'] <= lim_sup)].copy()

print('Media antes:', round(df['total_bill'].mean(), 2))
print('Media depois:', round(df_sem['total_bill'].mean(), 2))
print('Linhas antes/depois:', len(df), len(df_sem))


Q1: 13.35 Q3: 24.13 IQR: 10.78
Limites: -2.82 a 40.3
Qtd outliers: 9
Media antes: 19.79
Media depois: 18.8
Linhas antes/depois: 244 235


## Exercício 2: Scaling na Prática

DataFrame com altura (m) e peso (g). Aplicar MinMaxScaler e StandardScaler e ver a distância euclidiana entre dois pontos antes e depois.


In [2]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from numpy.linalg import norm

df_pessoas = pd.DataFrame({
    'Altura': [1.60, 1.75, 1.82, 1.68],
    'Peso': [55000, 70000, 82000, 63000]
})
print(df_pessoas)

p0 = df_pessoas.iloc[0].values
p1 = df_pessoas.iloc[1].values
dist_original = norm(p0 - p1)
print('Distancia original (0 vs 1):', round(dist_original, 2))

minmax = MinMaxScaler()
df_minmax = pd.DataFrame(minmax.fit_transform(df_pessoas), columns=df_pessoas.columns)
print('MinMaxScaler:')
print(df_minmax.round(3))
dist_minmax = norm(df_minmax.iloc[0].values - df_minmax.iloc[1].values)
print('Distancia MinMax:', round(dist_minmax, 4))

std = StandardScaler()
df_std = pd.DataFrame(std.fit_transform(df_pessoas), columns=df_pessoas.columns)
print('StandardScaler:')
print(df_std.round(3))
dist_std = norm(df_std.iloc[0].values - df_std.iloc[1].values)
print('Distancia Standard:', round(dist_std, 4))


   Altura   Peso
0    1.60  55000
1    1.75  70000
2    1.82  82000
3    1.68  63000
Distancia original (0 vs 1): 15000.0
MinMaxScaler:
   Altura   Peso
0   0.000  0.000
1   0.682  0.556
2   1.000  1.000
3   0.364  0.296
Distancia MinMax: 0.8795
StandardScaler:
   Altura   Peso
0  -1.378 -1.261
1   0.459  0.252
2   1.316  1.463
3  -0.398 -0.454
Distancia Standard: 2.3799


Sem scaling, o peso (em gramas) domina a distância porque a escala é bem maior que a da altura. Depois do MinMax ou StandardScaler as duas variáveis ficam comparáveis.


## Exercício 3: Encoding Completo

Colunas: Cidade (SP, RJ, MG), Nivel (Junior, Pleno, Senior) e Genero (M, F). Aplicar o encoding adequado e justificar.


In [3]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

df_cat = pd.DataFrame({
    'Cidade': ['SP', 'RJ', 'MG', 'SP', 'RJ'],
    'Nivel': ['Junior', 'Pleno', 'Senior', 'Pleno', 'Junior'],
    'Genero': ['M', 'F', 'M', 'F', 'M']
})
print(df_cat)

ohe_cidade = OneHotEncoder(sparse_output=False, dtype=int)
cidade_arr = ohe_cidade.fit_transform(df_cat[['Cidade']])
df_cidade = pd.DataFrame(cidade_arr, columns=ohe_cidade.get_feature_names_out(['Cidade']))

ord_nivel = OrdinalEncoder(categories=[['Junior', 'Pleno', 'Senior']])
df_cat['Nivel_ord'] = ord_nivel.fit_transform(df_cat[['Nivel']]).astype(int)

df_cat['Genero_bin'] = df_cat['Genero'].map({'F': 0, 'M': 1})

df_final = pd.concat([df_cidade, df_cat[['Nivel_ord', 'Genero_bin']]], axis=1)
print('Depois do encoding:')
print(df_final)


  Cidade   Nivel Genero
0     SP  Junior      M
1     RJ   Pleno      F
2     MG  Senior      M
3     SP   Pleno      F
4     RJ  Junior      M
Depois do encoding:
   Cidade_MG  Cidade_RJ  Cidade_SP  Nivel_ord  Genero_bin
0          0          0          1          0           1
1          0          1          0          1           0
2          1          0          0          2           1
3          0          0          1          1           0
4          0          1          0          0           1


Justificativa:
- Cidade: sem ordem -> One-Hot
- Nivel: Junior < Pleno < Senior -> OrdinalEncoder
- Genero: duas classes nominais -> 0/1
